In [23]:
import pandas as pd
import numpy as np

np.set_printoptions(threshold=np.inf)

In [24]:
# Loading guest data from the restaurant
guest_df = pd.read_csv("../data/WEEVA_GUESTS.csv")
guest_df.head(5)
# guest_df.shape

,DATE,GUESTS
0,11/1/2018,0
1,11/2/2018,0
2,11/3/2018,0
3,11/4/2018,0
4,11/5/2018,108


In [25]:
# Loading weather data by time frame
weather_nov_2018_may_2021_df       = pd.read_csv("../data/Groningen 2018-11-01 to 2021-05-31.csv")
weather_june_2021_december_2023_df = pd.read_csv("../data/Groningen 2021-06-01 to 2023-12-31.csv")
weather_jan_2024_april_2025_df     = pd.read_csv("../data/Groningen 2024-01-01 to 2025-04-28.csv")

# Combining the weather datasets into one dataframe
weather_df = pd.concat([weather_nov_2018_may_2021_df, 
                        weather_june_2021_december_2023_df, 
                        weather_jan_2024_april_2025_df], 
                        ignore_index=True)

weather_df.head()
weather_df.shape

(2371, 33)

In [26]:
# Loading data on holidays and calendar dates
school_holidays_df           = pd.read_csv("../data/groningen_school_holidays_boolean.csv")
public_holidays_groningen_df = pd.read_csv("../data/public_holidays_2018_2025.csv")
public_holidays_germany_df   = pd.read_csv("../data/public_holidays_germany_2018_2025.csv")
calendar_df                  = pd.read_csv("../data/dates_with_weekdays.csv")

calendar_df.head()

,Date,DayOfWeek,IsWeekend
0,2018-11-01,Thursday,False
1,2018-11-02,Friday,False
2,2018-11-03,Saturday,True
3,2018-11-04,Sunday,True
4,2018-11-05,Monday,False


In [27]:
school_holidays_df.tail()
school_holidays_df.shape
# school_holidays_df.dtypes

(3035, 2)

In [28]:
school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])

start_date = '2018-11-01'
end_date = '2025-04-28'

# Filter to keep only dates within the desired range
school_holidays_df = school_holidays_df[
    (school_holidays_df['Date'] >= start_date) &
    (school_holidays_df['Date'] <= end_date)
]

# Remove duplicate dates, keeping the last occurrence
school_holidays_df = school_holidays_df.drop_duplicates(subset='Date', keep='last')

school_holidays_bool_df = pd.DataFrame(school_holidays_df)
# Convert 'Yes'/'No' to True/False in a specific column (e.g., 'IsHoliday')
school_holidays_bool_df['IsHoliday'] = school_holidays_bool_df['IsHoliday'].map({"Yes": True, "No": False})

school_holidays_bool_df.shape

C:\Users\Matei\AppData\Local\Temp\ipykernel_6812\819521641.py:1: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  school_holidays_df["Date"] = pd.to_datetime(school_holidays_df["Date"])


(2371, 2)

In [29]:
# 9 entries out of our date range for groningen
# public_holidays_groningen_df.head(50)
# public_holidays_groningen_df.tail(20)

public_holidays_groningen_df["Holiday"].unique()
# Only 9 unique holidays, but inconsistant naming () 
# e.g: 'Koningsdag (National Holiday)' / 'Koningsdag (National Day)'
# public_holidays_groningen_df["Holiday"].unique().shape

public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df\
                                                       ['Holiday'].str.strip()

# public_holidays_groningen_df["Holiday"].unique()

name_map = {
    "New Year": "New Year's Day",
    "Koningsdag (National Holiday)": "King\'s Day",
     "Koningsdag (National Day)": "King\'s Day",
     "St. Stephen's Day": "Second Christmas Day"
}

# Solve inconsistant naming
public_holidays_groningen_df["Holiday"] = public_holidays_groningen_df\
                                                 ['Holiday'].replace(name_map)

public_holidays_groningen_df["Holiday"].unique()
# # Only 7 unique holidays after filtering
# public_holidays_groningen_df["Holiday"].unique().shape

# public_holidays_groningen_df.head()


array(["New Year's Day", 'Easter Monday', "King's Day", 'Ascension Day',
       'Whit Monday', 'Christmas', 'Second Christmas Day'], dtype=object)

In [30]:
public_holidays_germany_df["Holiday"].unique()

public_holidays_germany_df["Holiday"] = public_holidays_germany_df\
                                                       ['Holiday'].str.strip()

name_map = {
    "New Years Day": "New Year's Day",
    "Christmas Day": "Christmas",
    "Boxing Day": "Second Christmas Day"
}

# Solve inconsistant naming
public_holidays_germany_df["Holiday"] = public_holidays_germany_df\
                                                 ['Holiday'].replace(name_map)

public_holidays_germany_df["Holiday"].unique()


array(["New Year's Day", 'Good Friday', 'Easter Monday', 'May Day',
       'Ascension Day', 'Whit Monday', 'Day of German Unity', 'Christmas',
       'Second Christmas Day'], dtype=object)

In [31]:
# Setting the date column to the right data type
public_holidays_groningen_df["Date"] = pd.to_datetime(
                                        public_holidays_groningen_df["Date"],
                                        format="%d.%m.%Y")
public_holidays_germany_df["Date"] = pd.to_datetime(
                                        public_holidays_germany_df["Date"],
                                        format="%d.%m.%Y")

# Combine all the holidays
combined_holidays_df = pd.concat([public_holidays_groningen_df,
                                   public_holidays_germany_df],
                                    ignore_index=True)

combined_holidays_df["Holiday"].unique().shape

combined_holidays_df['is_holiday'] = 1


# Drop duplicates
combined_holidays_df = combined_holidays_df.pivot_table(
    index='Date',
    columns='Holiday',
    values='is_holiday',
    fill_value=0
).reset_index()

combined_holidays_df.shape

# Defining a range of dates for the full holiday dataframe
date_range = pd.date_range(start='2018-11-01', end='2025-04-28', freq='D')

# Create a new DataFrame with that full date range
full_date_range_df = pd.DataFrame({'Date': date_range})

# Merge on Date — left join to preserve full date range
merged_df = full_date_range_df.merge(combined_holidays_df,
                                     on='Date',
                                     how='left')

# # Fill NaN's 
final_holiday_df = merged_df.fillna('0').astype({col: 'int' for col in \
                                                     merged_df.columns \
                                                        if col != 'Date'})

final_holiday_df.head()
final_holiday_df.columns




Index(['Date', 'Ascension Day', 'Christmas', 'Day of German Unity',
       'Easter Monday', 'Good Friday', 'King's Day', 'May Day',
       'New Year's Day', 'Second Christmas Day', 'Whit Monday'],
      dtype='object')

In [32]:
calendar_df.head()
calendar_df.shape

(2371, 3)

In [ ]:
# Loading data on number of items ordered in the restaurant
course_data_2018_2022_df = pd.read_csv("../data/Weeva_data_2018-2022.csv")
course_data_2023_2025_df = pd.read_csv("../data/Weeva_Data_2023-x.csv")

course_data_df = pd.concat([course_data_2018_2022_df, 
                            course_data_2023_2025_df], 
                            ignore_index=True)

# Setting the date column to the right data type
course_data_df["Date"] = pd.to_datetime(course_data_df["Date"],
                                        format="%d-%m-%Y")

article_map = {
    "broodplankje": "art_broodplankje",
    "captain.*dinner": "art_captain_dinner",
    "(?<!\w\s)cr.me.*br.l.e(?!\s)": "art_creme_brulee",
    "dame.*blanche(?!\s)": "art_dame_blanche",
    "sliptong.*meuni.re": "art_sliptong",
    "garnalen.*cocktail": "art_garnalen_cocktail",
    "bloedworst": "art_bloedworst",
    "olijven": "art_olijven",
    "kaasplankje(?!\s)": "art_kaasplankje",
    "(?<!\w\s)kalfslever": "art_kalfslever",
    "koffie.*compleet(?!\s)": "art_koffie_compleet",
    "groningse poffert": "art_poffert",
    "runder.*carpaccio": "art_carpaccio",
    "sat.*spies(?!\s)": "art_sate_spies",
    "schnitzel": "art_schnitzel",
    "sorbet.*weeva": "art_sorbet",
    "andijvie stamppot": "art_stamppot",
    "vers.*markt": "art_vers_van_de_markt",
    "weeva.*gehaktbal(?!\s)": "art_gehaktbal",
    "weeva.*spareribs(?!\s)": "art_spareribs",
    "tournedos(?!\s)": "art_tournedos",
    "zalmfilet(?!\s)": "art_zalmfilet",
}

# Lower case
course_data_df["Article"] = course_data_df["Article"].str.lower()

# Solve inconsistant naming
course_data_df["Article"] = course_data_df["Article"].replace(article_map,\
                                                               regex=True)

# Keep only rows where the Article value is one of the mapped values
valid_articles = set(article_map.values())
course_data_df = course_data_df[course_data_df["Article"].\
                                                        isin(valid_articles)]

# print(course_data_df["Article"].unique())
# print(course_data_df.head())

course_data_df = course_data_df.pivot_table(
    index="Date",
    columns="Article",
    values="Sold articles amount",
    fill_value=0,
    aggfunc="sum",
    ).reset_index()

print(course_data_df.head())


#TODO: filter date range and finish todo list.md


Article       Date  art_bloedworst  art_broodplankje  art_captain_dinner  \
0       2018-11-27               0                 3                   0   
1       2018-11-28               0                 7                   0   
2       2018-11-29               4                 9                   0   
3       2018-11-30               3                12                   0   
4       2018-12-01               7                13                   0   

Article  art_carpaccio  art_creme_brulee  art_dame_blanche  \
0                   11                 0                 5   
1                    3                 7                 3   
2                    7                10                11   
3                    5                 5                14   
4                   10                 5                15   

Article  art_garnalen_cocktail  art_gehaktbal  art_kaasplankje  ...  \
0                            2              0                0  ...   
1                           